# Pré-processamento automático de registos de EEG

O notebook `01_remocao_artefactos.ipynb` descreve, passo a passo, os fundamentos da remoção de artefactos. O presente notebook aplica esses mesmos passos de forma **automática** a registos próprios, produzindo um registo corrigido e um relatório.

### Etapas do processamento
1. **Leitura** do registo (formatos `.edf`, `.bdf`, `.fif`, `.set`, `.vhdr`, entre outros)
2. **Remoção da interferência da rede elétrica** (50 ou 60 Hz, com deteção automática)
3. **Filtragem** das oscilações lentas e do ruído de alta frequência
4. **Deteção de elétrodos com mau contacto** e respetiva interpolação
5. **Análise de Componentes Independentes (ICA)**, com classificação automática dos componentes por um modelo treinado (ICLabel): atividade cerebral, ocular, muscular, cardíaca, entre outras
6. **Exclusão dos componentes de origem não cerebral**
7. **Exportação** do registo corrigido e de um relatório (página HTML com todas as figuras), para descarregar

### Utilização
Execute os blocos pela ordem apresentada (ou **Runtime → Run all**). No bloco **Carregamento dos registos** é apresentado um botão para selecionar os ficheiros. Na ausência de ficheiros, é utilizado um registo de exemplo.

## 0. Preparação

In [ ]:
# Instalação, no ambiente do Colab, das bibliotecas necessárias:
#   mne          - leitura, processamento e visualização de sinais de EEG
#   mne-icalabel - classificador automático de componentes ICA (modelo ICLabel)
#   onnxruntime  - motor de cálculo utilizado pelo modelo ICLabel
#   pyprep       - deteção de elétrodos com mau contacto (método PREP)
# A instalação demora cerca de um minuto; o relatório detalhado só é apresentado se ocorrer um erro.
import subprocess, sys
print("A instalar as bibliotecas...")
result = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                         "mne", "mne-icalabel", "onnxruntime", "pyprep"],
                        capture_output=True, text=True)
print("Instalação concluída." if result.returncode == 0 else result.stderr)

In [ ]:
# Bibliotecas utilizadas no processamento
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd                                   # tabelas de resultados
import mne
from mne.datasets import eegbci                       # registo de exemplo (PhysioNet)
from mne_icalabel import label_components             # classificação dos componentes ICA
from pyprep.find_noisy_channels import NoisyChannels  # deteção de elétrodos com mau contacto
from google.colab import files                        # carregamento e descarga de ficheiros no Colab

mne.set_log_level("ERROR")                 # o MNE apresenta apenas mensagens de erro
mne.viz.set_browser_backend("matplotlib")  # os sinais são apresentados como imagens estáticas
plt.rcParams["figure.dpi"] = 100           # resolução das figuras

RAW_DIR = Path("data/raw")            # registos originais
OUT_DIR = Path("data/preprocessed")   # resultados do processamento
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Carregamento dos registos

Ao executar o bloco seguinte, é apresentado o botão **"Choose Files"**, que permite selecionar um ou mais registos.
- **BrainVision** (`.vhdr`): selecione também os ficheiros `.vmrk` e `.eeg` com o mesmo nome.
- **EEGLAB** (`.set`): se existir um ficheiro `.fdt` com o mesmo nome, selecione-o também.
- Para utilizar apenas o **registo de exemplo**, selecione **Cancel upload**.

In [ ]:
# Extensões dos ficheiros de registo principais. Os ficheiros auxiliares (.vmrk, .eeg, .fdt)
# são guardados, mas não contam como registos independentes.
MAIN_EXTENSIONS = {".edf", ".bdf", ".fif", ".set", ".vhdr", ".cnt", ".gdf"}

# Abrir a janela de seleção de ficheiros e mover os ficheiros recebidos para a pasta data/raw/
uploaded = files.upload()
for fname in uploaded:
    Path(fname).rename(RAW_DIR / fname)

MY_FILES = [RAW_DIR / f for f in uploaded if Path(f).suffix.lower() in MAIN_EXTENSIONS]
print("Registos carregados:", [f.name for f in MY_FILES] or "nenhum (será utilizado o registo de exemplo)")

## Parâmetros

Os valores predefinidos são adequados para a maioria dos registos.

In [ ]:
# --- Elétrodos ---------------------------------------------------------------
# Montagem padrão utilizada para atribuir posições aos elétrodos, quando o ficheiro não as inclui
MONTAGE = "standard_1005"

# --- Filtragem ---------------------------------------------------------------
LINE_FREQ = None   # frequência da rede elétrica: 50, 60, ou None para deteção automática
L_FREQ = 1.0       # frequência de corte inferior (Hz): remove oscilações lentas da linha de base
H_FREQ = 40.0      # frequência de corte superior (Hz) do registo final

# --- Elétrodos com mau contacto ----------------------------------------------
# Critérios do método PREP (Bigdely-Shamlo et al., 2015). A remoção de critérios da lista
# torna a deteção menos exigente.
BAD_CHANNEL_CRITERIA = ["bad_by_nan", "bad_by_flat", "bad_by_deviation", "bad_by_hf_noise",
                        "bad_by_correlation", "bad_by_SNR", "bad_by_dropout", "bad_by_ransac"]

# --- ICA ---------------------------------------------------------------------
N_COMPONENTS = 20   # número de componentes independentes a estimar
# Tipos de componente a excluir e probabilidade mínima atribuída pelo classificador (0 a 1)
REMOVE_LABELS = ["eye blink", "muscle artifact", "heart beat", "line noise", "channel noise"]
MIN_PROBABILITY = 0.80

## Funções de processamento

O bloco seguinte define cada etapa do processamento como uma função. **Não é necessário alterá-lo**; basta executá-lo.

In [ ]:
# Tradução das categorias do classificador ICLabel
LABELS_PT = {"brain": "cerebral", "eye blink": "ocular", "muscle artifact": "muscular",
             "heart beat": "cardíaco", "line noise": "rede elétrica", "channel noise": "ruído de elétrodo",
             "other": "outro"}


def load_recording(fname=None):
    # Lê um registo a partir do ficheiro indicado. Sem ficheiro, descarrega um registo de exemplo
    # da base de dados PhysioNet EEG Motor Movement/Imagery (voluntário 1, sessão 4).
    if fname is None:
        fname = eegbci.load_data(1, [4], path=RAW_DIR, update_path=False)[0]
    raw = mne.io.read_raw(fname, preload=True)
    if raw.get_montage() is None:
        set_positions(raw)
    return raw, Path(fname).name.split(".")[0]


def set_positions(raw):
    # Atribui a cada elétrodo a posição correspondente na montagem padrão.
    # Os nomes dos elétrodos variam entre equipamentos ("FP1", "Fp1.", "EEG Fz", ...), pelo que
    # são primeiro convertidos para a nomenclatura padrão ("Fp1", "Fz", ...).
    standard = mne.channels.make_standard_montage(MONTAGE)
    lookup = {ch.lower(): ch for ch in standard.ch_names}
    rename = {}
    for ch in raw.ch_names:
        key = ch.strip(" .").lower().removeprefix("eeg ").strip()
        if key in lookup and lookup[key] not in raw.ch_names:
            rename[ch] = lookup[key]
    raw.rename_channels(rename)
    raw.set_montage(standard, on_missing="ignore")
    # Os elétrodos sem posição conhecida não podem ser interpolados nem incluídos na ICA,
    # pelo que são reclassificados como canais auxiliares ("misc") e excluídos do processamento.
    unknown = [c["ch_name"] for c in raw.info["chs"]
               if c["kind"] == mne.io.constants.FIFF.FIFFV_EEG_CH and np.isnan(c["loc"][:3]).any()]
    if unknown:
        raw.set_channel_types({ch: "misc" for ch in unknown})
        print(f"Atenção: elétrodos sem posição conhecida, excluídos do processamento: {', '.join(unknown)}")


def detect_line_freq(raw):
    # Deteta a interferência da rede elétrica: procura um pico estreito no espectro aos 50 ou 60 Hz,
    # pelo menos 3 dB acima das frequências vizinhas. Devolve 50, 60 ou None.
    nyquist = raw.info["sfreq"] / 2
    psd = raw.compute_psd(picks="eeg", fmin=40, fmax=min(70, nyquist - 1))
    power = 10 * np.log10(psd.get_data().mean(axis=0))
    best, best_peak = None, 3.0
    for f in (50, 60):
        if f < nyquist - 1:
            dist = np.abs(psd.freqs - f)
            peak = power[dist < 0.6].max() - np.median(power[(dist > 2) & (dist < 6)])
            if peak > best_peak:
                best, best_peak = f, peak
    return best


def filter_recording(raw, line_freq):
    # 1) Filtro notch: remove a frequência da rede elétrica e as suas harmónicas (ex.: 50, 100, 150 Hz).
    # 2) Filtro passa-banda de L_FREQ a 100 Hz (ou ao máximo permitido pela frequência de amostragem),
    #    a gama de frequências com que o classificador ICLabel foi treinado.
    data = raw.copy()
    if line_freq:
        data.notch_filter(np.arange(line_freq, raw.info["sfreq"] / 2, line_freq), picks="eeg")
    return data.filter(l_freq=L_FREQ, h_freq=min(100.0, raw.info["sfreq"] / 2 - 5), picks="eeg")


def find_bad_channels(raw):
    # Aplica os critérios do método PREP e devolve, para cada elétrodo com mau contacto,
    # a lista de critérios que o identificaram (ex.: {"T7": ["hf_noise"]}).
    nc = NoisyChannels(raw.copy().pick("eeg"), random_state=97)
    nc.find_all_bads(ransac="bad_by_ransac" in BAD_CHANNEL_CRITERIA)
    found = nc.get_bads(as_dict=True)
    return {ch: [c.removeprefix("bad_by_") for c in BAD_CHANNEL_CRITERIA if ch in found.get(c, [])]
            for ch in sorted({ch for c in BAD_CHANNEL_CRITERIA for ch in found.get(c, [])})}


def run_ica(raw):
    # Decompõe o sinal em componentes independentes e classifica cada um com o ICLabel.
    # O ICLabel foi treinado com o algoritmo "extended infomax", referência média e filtro 1-100 Hz,
    # pelo que estas condições são reproduzidas aqui.
    n_good = len(mne.pick_types(raw.info, eeg=True, exclude="bads"))
    ica = mne.preprocessing.ICA(n_components=min(N_COMPONENTS, n_good - 1), method="infomax",
                                fit_params=dict(extended=True), random_state=97, max_iter="auto")
    ica.fit(raw, picks="eeg")
    result = label_components(raw, ica, method="iclabel")
    labels, probs = result["labels"], result["y_pred_proba"]
    # Componentes a excluir: categoria em REMOVE_LABELS e probabilidade de pelo menos MIN_PROBABILITY
    remove = [i for i, (l, p) in enumerate(zip(labels, probs)) if l in REMOVE_LABELS and p >= MIN_PROBABILITY]
    table = pd.DataFrame({"componente": [f"ICA{i:03d}" for i in range(len(labels))],
                          "classificação": [LABELS_PT[l] for l in labels],
                          "probabilidade": np.round(probs, 2),
                          "excluir": ["sim" if i in remove else "" for i in range(len(labels))]})
    return ica, remove, table


def save_results(name, raw, before_ica, cleaned, ica, removed, table, bads, line_freq):
    # Guarda o registo corrigido (.fif), a decomposição ICA (.fif) e um relatório HTML
    # com o resumo das decisões e as figuras do registo original, da ICA e do registo corrigido.
    paths = [OUT_DIR / f"{name}_clean-raw.fif", OUT_DIR / f"{name}-ica.fif", OUT_DIR / f"{name}_relatorio.html"]
    cleaned.save(paths[0], overwrite=True)
    ica.save(paths[1], overwrite=True)
    report = mne.Report(title=f"Pré-processamento do EEG: {name}")
    report.add_html(f"<p><b>Interferência da rede elétrica:</b> {line_freq or 'não detetada'} Hz<br>"
                    f"<b>Elétrodos interpolados:</b> {', '.join(bads) or 'nenhum'}<br>"
                    f"<b>Componentes excluídos:</b> {', '.join(f'ICA{i:03d}' for i in removed) or 'nenhum'}</p>"
                    + table.to_html(index=False), title="Resumo")
    report.add_raw(raw, title="Registo original", psd=True, butterfly=False)
    report.add_ica(ica, title="ICA", inst=before_ica, n_jobs=1)
    report.add_raw(cleaned, title="Registo corrigido", psd=True, butterfly=False)
    report.save(paths[2], overwrite=True, open_browser=False)
    return paths


def clean_recording(fname):
    # Aplica todas as etapas a um registo, sem figuras intermédias (utilizada no processamento em lote).
    raw, name = load_recording(fname)
    line_freq = LINE_FREQ or detect_line_freq(raw)
    data = filter_recording(raw, line_freq)
    bads = list(find_bad_channels(data))
    data.info["bads"] = bads
    if bads:
        data.interpolate_bads()
    data.set_eeg_reference("average")
    ica, remove, table = run_ica(data)
    ica.exclude = remove
    cleaned = ica.apply(data.copy()).filter(l_freq=None, h_freq=H_FREQ, picks="eeg")
    paths = save_results(name, raw, data, cleaned, ica, remove, table, bads, line_freq)
    return dict(name=name, line_freq=line_freq, bads=bads, removed=remove, paths=paths)


def show(recordings, titles, channels=None, start=30, duration=10):
    # Representa o mesmo intervalo de tempo de vários registos, uns por baixo dos outros.
    # Por omissão, mostra 12 elétrodos distribuídos pelo escalpe, de anterior para posterior.
    if channels is None:
        eeg = recordings[0].copy().pick("eeg")
        by_position = [c["ch_name"] for c in sorted(eeg.info["chs"], key=lambda c: -c["loc"][1])]
        channels = [by_position[i] for i in np.linspace(0, len(by_position) - 1, 12).astype(int)]
    start = min(start, recordings[0].times[-1] - duration)
    fig, axes = plt.subplots(len(recordings), 1, sharex=True, squeeze=False,
                             figsize=(12, 0.4 * len(channels) * len(recordings) + 1.5))
    spacing = None
    for ax, rec, title in zip(axes[:, 0], recordings, titles):
        data, times = rec.get_data(picks=channels, tmin=start, tmax=start + duration, return_times=True)
        data = (data - data.mean(axis=1, keepdims=True)) * 1e6   # remover o valor médio e converter V -> µV
        spacing = spacing or 4 * np.median(np.std(data, axis=1))  # mesma escala vertical em todas as figuras
        for i, trace in enumerate(data):
            ax.plot(times, trace - i * spacing, color="black", lw=0.6)
        ax.set_yticks(-np.arange(len(channels)) * spacing, channels)
        ax.set_ylim(-len(channels) * spacing, 1.5 * spacing)
        ax.set_title(title, loc="left", fontweight="bold")
        ax.spines[["top", "right", "left"]].set_visible(False)
    axes[-1, 0].set_xlabel("Tempo (segundos)")
    axes[-1, 0].set_xlim(times[0], times[-1])
    plt.tight_layout()
    plt.show()


def spectrum(recordings, labels, title):
    # Compara a densidade espectral de potência (média de todos os elétrodos) de vários registos
    fig, ax = plt.subplots(figsize=(10, 3.5))
    for rec, label in zip(recordings, labels):
        psd = rec.compute_psd(picks="eeg")
        ax.plot(psd.freqs, 10 * np.log10(psd.get_data().mean(axis=0) * 1e12), label=label)
    ax.set_xlabel("Frequência (Hz)")
    ax.set_ylabel("Potência (dB)")
    ax.set_title(title, loc="left", fontweight="bold")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()


def download(paths, zip_name):
    # Agrupa os ficheiros indicados num arquivo .zip e inicia a sua descarga no navegador
    zip_path = OUT_DIR / zip_name
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in paths:
            z.write(p, arcname=p.name)
    files.download(str(zip_path))

print("Funções definidas.")

---
# Processamento passo a passo de um registo

As secções seguintes aplicam cada etapa a **um** registo (o primeiro carregado ou, na ausência de ficheiros, o registo de exemplo), apresentando os resultados intermédios.

## 1. Leitura do registo

In [ ]:
raw, name = load_recording(MY_FILES[0] if MY_FILES else None)

n_eeg = len(mne.pick_types(raw.info, eeg=True))
print(f"Registo                  : {name}")
print(f"Elétrodos de EEG         : {n_eeg}  (outros canais: {len(raw.ch_names) - n_eeg})")
print(f"Duração                  : {raw.times[-1]:.0f} segundos")
print(f"Frequência de amostragem : {raw.info['sfreq']:.0f} Hz")

# Disposição dos elétrodos no escalpe, para verificação
raw.plot_sensors(show_names=True, sphere="eeglab");

**Verificação:** confirme que os elétrodos surgem nas posições corretas. Se o registo incluir canais que não são de EEG (por exemplo, `EOG` ou `ECG`) mas que surgem classificados como EEG, indique o tipo correto neste bloco e execute-o novamente, por exemplo: `raw.set_channel_types({"VEOG": "eog", "ECG": "ecg"})`.

## 2. Interferência da rede elétrica e filtragem

In [ ]:
# Utilizar a frequência definida nos parâmetros ou, se não estiver definida, detetá-la no espectro
line_freq = LINE_FREQ or detect_line_freq(raw)
print("Interferência da rede elétrica:", f"{line_freq} Hz" if line_freq else "não detetada")

# Filtro notch na frequência da rede e filtro passa-banda (ver função filter_recording)
filtered = filter_recording(raw, line_freq)
spectrum([raw, filtered], ["original", "filtrado"], "Densidade espectral de potência antes e depois da filtragem")

Nesta etapa, o filtro estende-se até aos 100 Hz (ou ao limite permitido pela frequência de amostragem), uma vez que o classificador utilizado na etapa 4 foi treinado nessas condições. O corte final nos **40 Hz** (`H_FREQ`) é aplicado no fim do processamento.

## 3. Elétrodos com mau contacto

In [ ]:
# Identificar os elétrodos com mau contacto e o(s) critério(s) que os identificaram
bads = find_bad_channels(filtered)
for ch, reasons in bads.items():
    print(f"  {ch:5s}  critério(s): {', '.join(reasons)}")
print(f"{len(bads)} elétrodo(s) com mau contacto." if bads else "Não foram identificados elétrodos com mau contacto.")

Significado dos critérios: **flat**: sinal plano; **deviation**: amplitude muito diferente da dos restantes elétrodos; **hf_noise**: ruído de alta frequência excessivo (frequentemente de origem muscular); **correlation** / **ransac**: baixa concordância com os elétrodos vizinhos; **SNR**: baixa relação sinal-ruído; **dropout**: perdas intermitentes de sinal.

Os elétrodos identificados são interpolados a partir dos elétrodos vizinhos:

In [ ]:
repaired = filtered.copy()
repaired.info["bads"] = list(bads)
if bads:
    repaired.interpolate_bads()   # interpolação esférica a partir dos elétrodos vizinhos
    show([filtered, repaired], ["Elétrodos com mau contacto (sinal original)", "Após interpolação"],
         channels=list(bads)[:12])

# Referência média: o potencial de cada elétrodo passa a ser expresso em relação à média
# de todos os elétrodos (condição necessária para o classificador ICLabel)
repaired.set_eeg_reference("average");

## 4. ICA e classificação automática dos componentes

A ICA decompõe o sinal em componentes independentes. Cada componente é depois classificado pelo **[ICLabel](https://mne.tools/mne-icalabel/)**, um modelo treinado com milhares de componentes classificados por especialistas, nas categorias: cerebral, ocular, muscular, cardíaco, rede elétrica, ruído de elétrodo ou outro.

In [ ]:
# Decomposição, classificação e tabela de resultados (ver função run_ica)
ica, remove, table = run_ica(repaired)
table

In [ ]:
# Distribuição topográfica de cada componente
ica.plot_components(sphere="eeglab");
print("Componentes a excluir:", [f"ICA{i:03d}" for i in remove] or "nenhum")

**Recomenda-se a verificação da classificação automática.** Em caso de discordância, a seleção pode ser definida manualmente antes de prosseguir, por exemplo `remove = [0, 4]`. Para examinar um componente em detalhe: `ica.plot_properties(repaired, picks=[0])`.

## 5. Exclusão dos componentes e comparação

In [ ]:
# Excluir os componentes selecionados, reconstruir o sinal e aplicar o filtro final (H_FREQ)
ica.exclude = remove
cleaned = ica.apply(repaired.copy()).filter(l_freq=None, h_freq=H_FREQ, picks="eeg")

# Para uma comparação equivalente, o sinal "antes" recebe apenas a filtragem e a referência média
only_filtered = raw.copy().filter(L_FREQ, H_FREQ, picks="eeg").set_eeg_reference("average")
show([only_filtered, cleaned], ["Antes (apenas filtrado)", "Depois (processamento completo)"])
spectrum([raw, cleaned], ["original", "corrigido"], "Densidade espectral de potência: original vs. corrigido")

A potência do registo corrigido é globalmente inferior, em resultado da remoção dos artefactos (de grande amplitude) e da utilização da referência média. A atenuação acima dos 40 Hz corresponde ao filtro final.

## 6. Exportação dos resultados

In [ ]:
# Guardar os resultados e descarregá-los num único arquivo .zip
paths = save_results(name, raw, repaired, cleaned, ica, remove, table, list(bads), line_freq)
download(paths, f"{name}_resultados.zip")

O arquivo `.zip` contém:
- **`..._clean-raw.fif`**: o registo corrigido, pronto para análise (pode ser lido com `mne.io.read_raw("...")`);
- **`..._relatorio.html`**: o relatório com todas as figuras e decisões, que pode ser aberto em qualquer navegador;
- **`...-ica.fif`**: a decomposição ICA, que pode ser reutilizada sem necessidade de novo cálculo.

O ficheiro é guardado na pasta de transferências do navegador.

---
# Processamento em lote

Uma vez validados os parâmetros, o bloco seguinte aplica o processamento completo a **todos os registos carregados** e descarrega os resultados num único arquivo `.zip`.

In [ ]:
summary, all_paths = [], []
for f in MY_FILES:
    print(f"A processar {f.name}...")
    res = clean_recording(f)   # todas as etapas, sem figuras intermédias
    all_paths += res["paths"]
    summary.append({"ficheiro": f.name, "rede elétrica (Hz)": res["line_freq"],
                    "elétrodos interpolados": ", ".join(res["bads"]) or "-",
                    "componentes excluídos": ", ".join(f"ICA{i:03d}" for i in res["removed"]) or "-"})

if MY_FILES:
    download(all_paths, "todos_os_resultados.zip")
    display(pd.DataFrame(summary))   # tabela-resumo do processamento
else:
    print("Não foram carregados registos na secção Carregamento dos registos.")